LEFT JOIN vs INNER JOIN
    When choosing a JOIN, the question to ask is what is my population
        INNER JOIN
            Only entities that have matches
        LEFT JOIN
            Everyone on the left, matched where possible
    Analytics almost always starts with a population (users, orders, accounts)
    Population shouldn't accidentally shrink

Example 1: INNER JOIN (population shrinks)
Orders that contain at least one item with purchased products

SELECT
    o.order_id, #attribute from order table
    i.product #attribute from order items table
FROM orders o #order alias o
INNER JOIN order_items i #order items alias i
    ON o.order_id = i.order_id; #matches columns across order and order items tables

Example 2: LEFT JOIN (population preserved)
All orders and corresponding products, even if a section is NULL (like a product isn't purchased)

SELECT
    o.order_id,
    i.product
FROM orders o
LEFT JOIN order_items i
    ON o.order_id = i.order_id;

Example 3: LEFT JOIN with Aggregation
Amount of items for each order

SELECT
  o.order_id,
  COUNT(i.item_id) AS item_count
FROM orders o
LEFT JOIN order_items i
  ON o.order_id = i.order_id
GROUP BY o.order_id;

Example 4: INNER JOIN with Aggregation
Amount of items for each order as long as the count isn't NULL (no purchases means it will be excluded)

SELECT
  o.order_id,
  COUNT(i.item_id) AS item_count
FROM orders o
INNER JOIN order_items i
  ON o.order_id = i.order_id
GROUP BY o.order_id;

Double Counting
    Happens when
        Tables are JOINed
        Multiple rows are JOINed
        Rows are aggregated as if they were not multiplied
    The root cause is where the grain of teh data changed, but the aggregation didn't

    Example 1: Grain Change
    Order 104 appears twice, meaning that before JOIN it was 1 row per order and after JOIN it was 1 row per order_item

        SELECT
            o.order_id,
            i.item_id
        FROM orders o
        LEFT JOIN order_items i
            ON o.order_id = i.order_id;

    Example 2: Classic Wrong Query
    Revenue belongs to the order, but since JOIN causes 1 row per order_item, Order 104's revenue appears twice, so SUM counts it twice

    SELECT
        SUM(o.revenue) AS total_revenue
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id;

    If every order had exactly one item, the result would look correct
    This bug survives testing, passes casual review, and fails silently in production

Fixing Double Counting (Restoring the Correct Grain)

To aggregate correctly
    Decide the grain necessary
    Force data back to that grain
    Then aggregate
There are two safe patterns
    Pattern A: Aggregate before Joining
        Use when the joined table is only for filtering or labeling

        Example 1: Total revenue from ordering (even NULL orders) without using JOIN
        SELECT SUM(revenue) as total_revenue
        FROM orders;
    Pattern B: Group back to the original grain after JOIN
        Use when JOIN is required for logic
        JOIN multiplies rows, GROUP BY collapses back to 1 row per order, and SUM becomes safe again

        Example 1: Total revenue from ordering (even NULL orders unless INNER JOIN is used)
        SELECT SUM(order_revenue) AS total_revenue
        FROM (
        SELECT
            o.order_id,
            o.revenue AS order_revenue
        FROM orders o
        LEFT JOIN order_items i #INNER JOIN takes away any order with 0 items, even if a user still paid
            ON o.order_id = i.order_id
        GROUP BY o.order_id
        );

        Example 2: Inner query has each order appear once, revenue appera once, and grain is restored
        SELECT
            o.order_id,
            o.revenue
        FROM orders o
        JOIN order_items i
            ON o.order_id = i.order_id
        GROUP BY o.order_id;

COUNT(*) vs. COUNT(column) vs. COUNT(DISTINCT)
    After JOINs
        Rows are not entities
        COUNTing rows answers how many rows, but what about how many entities
            DISTINCT helps this situation
    
    Example 1: Count(*) (almost never what you want)
    Counts order-items, NULL matches, and duplicates

    SELECT COUNT(*)
    From orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id;

    Example 2: Count(column) (still dangerous)
    Counts total items sold, not orders, so it is fine if that is what intended

    SELECT COUNT(i.item_id)
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id;

    Example 3: COUNT(DISTINCT) for Orders (entity-safe)
    Counts how many orders exist

    SELECT COUNT(DISTINCT o.order_id)
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id;

    Example 4: COUNT(DISTINCT) for Users (entity-safe)
    Counts how many users exist

    SELECT COUNT(DISTINCT o.user_id)
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id;

LEFT JOIN and WHERE Trap
    May occur because
        WHERE filters after the JOIN
        NULL values fail conditions
        LEFT JOIN collapses to INNER JOIN silently

    Example 1: Broken Query
    Orders with no items are removed, so LEFT JOIN has lost it's purpose
    Only shows monitor orders, not NULL ones

    SELECT o.order_id, i.product
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id
    WHERE i.product = 'Monitor';

    Example 2: Correct Placement (ON clause)
    Filtering happens during matching, orders without items still survive, and NULLs remain intact
    Shows which orders are for monitors and which ones are not

    SELECT o.order_id, i.product
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id
    AND i.product = 'Monitor'; #and instead of where so that users with other products are not erased

    Example 3: Another Safe Alternative
    Preserves orders, but can be more error prone rather than using AND (WHERE still has the ability to erase rows, which would ruin LEFT JOIN's point of keeping all rows)
    Shows which orders are for monitors and which ones didn't buy any products at all

    SELECT o.order_id, i.product
    FROM orders o
    LEFT JOIN order_items i
        ON o.order_id = i.order_id
    WHERE i.product = 'Monitor' OR i.product IS NULL; #where and or so that users that didn't buy products are not erased (kind of takes away from LEFT JOIN)

    Based on Examples 2 and 3, it shows that if conditions are needed for the right table, but all rows still want to be kept from the left table, a condition must be put inside the ON clause using AND, so it is applied during matching, not afterward with WHERE (applies condition after the JOIN, which can erase rows)

Population vs Attributes
    Always ask what defines who should appear in the result
        That table goes on the left
    Everything else adds detail and should not remove rows unless explicitly intended

    Example 1: Correct Population Thinking
    Users define population, orders and items are attributes, and users with no orders still appear
    All users and how many items they've purchased

    SELECT
        u.user_id,
        COUNT(i.item_id) AS items_purchased
    FROM users u
    LEFT JOIN orders o
        ON u.user_id = o.user_id #get to orders (attribute) from users (population)
    LEFT JOIN order_items i
        ON o.order_id = i.order_id #get to items (attribute) from orders (attribute)
    GROUP BY u.user_id;

    Population tables go on the LEFT, while Attribute tables are LEFT JOINed in
        This will help with population errors, where the query runs and the numbers look reasonable, but they are wrong since LEFT JOIN is cancelled out by attributes

Homework (lesson 3)

Problem 1: Population and Conditional Aggregation
    Question a
        For each country return
            Total users
            Number of users who have at least one completed order
            completion_rate = buyers_completed/total_users
        Requirements
            Include countries even if completion_rate is 0
            Output columns
                country
                total_users
                buyers_completed
                completion_rate

        Answer
        SELECT u.country AS country, --get the country as the population
	        COUNT(DISTINCT u.user_id) AS total_users, --get the user count for each country
	        COUNT(DISTINCT o.user_id) AS buyers_completed, --get the users who had orders
	    1.0 * COUNT(DISTINCT o.user_id) /COUNT(DISTINCT u.user_id) AS completion_rate --multiply by decimal to make REAL numbers
        FROM users u --from the left users population table
        LEFT JOIN orders o --join with the right orders attribute table
	        ON u.user_id = o.user_id --connect tables together
        AND status = 'completed' --make sure it's for those that have a completed status (but AND instead of WHERE so that incomplete users don't get deleted)
        GROUP BY u.country; --show grouping of each country (population)

Problem 2: Multi-Join
    Question a
        For each user_id
            Total users
            Number of distinct orders that contain 2 or order_items rows (multi-item orders)
        Requirements
            All users must appear (even if zeros)
            Output columns
                user_id
                completed_revenue
                multi_item_orders
        
        Answer




What SQL Rows Mean (Grain)
Grain
    What one row represents
    If you
        Sum revenue at the wrong grain
        Count users at the wrong grain
        Join tables without thinking about grain
            Double counting or under counting will occur
    
    Example 1a: users
    One row = one user

    SELECT * FROM users;

    Example 1b: orders
    One row = one order from a user

    SELECT * FROM orders;

    Example 1c: order_items
    One row = one item from an order
    
    SELECT * FROM order_items

    Example 2: Same user, multiple orders
    One user appears multiple times, as they placed multiple orders

    SELECT user_id, order_id
    FROM orders
    ORDER BY user_id;

    Always state the grain of the table out load before writing SQL

Checkpoint
Problem 1
    What is the grain of
        users
        orders
        order_items

    Answer
        One row is one user for users. One row is one user's order for orders. One row is one of the order's items for order_items.

Problem 2
    If one order has 3 rows in order_items, how many rows will appear after joining orders with order_items?

    Answer
        There would be 3 rows, as one order has 3 order_item rows.

Problem 3
    Why is summing orders.revenue dangerous after joining to order_items?

    Answer
        This is dangerous, because it would count the revenue three separate times, as it sees each item as a separate order with the same revenue rather than all items being combined to equal the one order's revenue.




JOINs as Row Multiplication (Not Column Addition)
Core Misconception
    A JOIN does not add columns from another table, but rather multiples rows based on matching relationships

What JOIN Really Does
    Answers how many rows on the right match each row on the left
        Then it creates one output row per match

    Example 1: Combining orders and order_items
        There is 1 order, and 3 matching item rows, so after combining the order row appears 3 times, once per item

        orders
            order_id: 101
            revenue: 50
        order_items
            order_id: 101, 101, 101
            item: Keyboard, Mouse, Monitor

One-Too-Many
    When one left row matches many right rows, row count increases

    Example 1
        Orders (left) for order_items (right)
        Users (left) for orders (right)

INNER JOIN vs LEFT JOIN
    INNER JOIN
        Keep only rows that match both sides
            Row count may shrink and population can disappear
    LEFT JOIN
        Keep every row from the left, even if there is no match
            Row count stays higher than the left table
            Missing matches produce NULLs on the right
            Does not prevent row multiplication, only preserves population

Why JOINs Break Aggregations
    JOINs can repeat order rows
    
    Example 1: Summing duplicate rows, not duplicate values
        Revenue belongs to the order grain
            JOIN orders to order_items and then SUM(revenue) would create duplicate rows, not duplicate values

Visual Summary
    JOIN
        LEFT table rows x matching RIGHT table rows = output rows
        If
            left row has 0 matches
                1 output row (with NULLs)
            left row has 1 match
                1 output row
            left row has 3 matches
                3 output rows

Checkpoint
    Problem 1
        If a user has 4 orders, how many rows appear after combining users -> orders?

        Answer
            4 rows appear, as there are 4 orders for the user.
    
    Problem 2
        If one of those orders has 3 items, how many rows represent that single order after combining orders -> order_items?

        Answer
            3 rows appear, as there are 3 items for the order.
    
    Problem 3
        Why does a LEFT JOIN not prevent double counting?

        Answer
            If a left table row has multiple matches, then it will create duplicate right table rows as the output.

    Problem 4
        Which type of relationship is more dangerous for analytics
            one-to-one
            one-to-many

        Answer
            one-to-many would be more dangerous, as duplicates may be creates if a row from the left table is equal to multiple things from the right table.

    Problem 5
        Why is it wrong to think of JOINs as "adding columns"?

        Answer
            JOINs are just repeating rows (duplicates) from the left table to match the right. It's not necessarily adding columns.

Aggregations After JOINs (What Breaks and Why)
    What Aggregation Assumes
        Aggregation functions (SUM, COUNT, AVG, etc.) assume that each row represents one unit of what is being aggregated
            If the assumption is violated, the result is wrong, as SQL only sees rows, not what the values mean

    What Changes after a JOIN
        After a JOIN
            The same logical entity can appear multiple times
            Aggregation functions now operate on duplicates rows

    SUM after JOINs
        Before JOIN
            orders
                order_id: 101
                revenue: 50
            SUM(revenue) = 50

        After JOIN with order_items (3 items)
            order_id: 101, 101, 101
            revenue: 50, 50, 50
            item: Keyboard, Mouse, Monitor
            SUM(revenue) = 150

        Revenue belongs to the order, so the order row was duplicated, and SUM blindly adds every row
        If the thing you're summing appears multiple times, it will be overcounted

    COUNT after JOINs
        COUNT(*)
            Counts rows, not entities
            After JOIN
                More rows -> higher count
                Even if it's the same order or user repeated
        COUNT(column)
            Counts non-NULL values, not unique entities
                If a column is repeated
                    It will be counted repeatedly
        Example 1
            If one user has 4 orders
                COUNT(user_id) = 4
            But if the question is "How many users?"
                The correct answer is 1, not 4
        COUNT does not know what an entity is, it only counts rows

    Why COUNT(DISTINCT) is not a Universal Fix
        SUM(DISTINCT revenue)
        COUNT(DISTINCT user_id)
            Removes duplicate values, not duplicate entities
            Two different orders with the same revenue values would be collapsed incorrectly
            Hides grain problems instead of fixing them
        Fixes symptoms, not causes
    
    AVG after JOINs
        AVG = SUM / COUNT
        If SUM or COUNT is inflated
            AVG can appear "reasonable", but still be wrong

    Diagnostic Checklist
        Before aggregating after JOIN, ask
            What the grain of the table is right now
            Does this value belong to that grain
            Can the same value appear more than once
            Am I counting rows or entities
            Would this still be correct if one entity had many matches

    Checkpoint
        Problem 1
            Why does SUM(revenue) break afater joining orders -> order_items?

            Answer
                Since one order has three items, that would mean that SUM(revenue) is counting the order three times instead of once

        Problem 2
            Why can COUNT(user_id) be misleading after joining users -> orders?

            Answer
                A user may have multiple orders, meaning that a user may be counted multiple times.

        Problem 3
            Why is COUNT(DISTINCT revenue) not a reliable fix for revenue duplication?

            Answer
                Duplicate values are removed, but not duplicate entites. So, two different orders with the same revenue values would be collapsed, leading to less entities than there actually should be.

        Problem 4
            Why can AVG(revenue) appear reasonable even when it is wrong?

            Answer
                The SUM or COUNT may be inflated, and since AVG relies on these values, the result could be either over (duplicates) or under (deleted entities from DISTINCT).

        Problem 5
            Given a joined table, what is the first question you should ask before using an aggregate?
                
            Answer
                What is the grain of the table right now?

Conditional Logic with CASE
    How to count and sum conidtionally without breaking population

    Core Problem CASE Solves
        Questions to answer
            How many completed orders?
            How much revenue from completed orders?
            How many users did X?
        WHERE would be first instinct, but removing rows changes
            Population
            Denominators
            Rates
            Averages
    
    Golden Rule
        Use WHERE to define the population
        Use CASE to define metrics within that population

    What is CASE
        SQL's version of if / else
        Evaluates row by row and returns a value
        Structure
            CASE
                When condition THEN value
                ELSE other_value
            END
        Does not remove rows, only changes what value a row contributes

    Why CASE is Safer than WHERE
        Example 1: For each user, how many completed orders did they place?
            Dangers
                Filter to completed orders
                Count orders
                    Users with zero completed orders disappear
                    Population has changed
            Safe
                Keep all users
                Count completed orders using logic
                Preserves
                    Users with 0
                    Correct denominators
                    Correct rates

    Counting with CASE
        Conditional count pattern
            SUM(CASE
                    WHEN condition THEN 1
                    ELSE 0
                END
            )
                Every row stays with no row removal
                Rows that meet the condition contribute 1
                Rows tha don't contribute 0

        Example 1: Sum of completed orders
            Each order is evaluated
            Completed adds 1
            Not completed adds 0
            SUM adds them up per user

            SELECT
                user_id,
                SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_orders
            FROM orders
            GROUP BY user_id;

    Conditional Sums
        SUM(
            CASE
                WHEN condition THEN value
                ELSE 0
            END
        )

        Example 1: Sum of revenue from completed orders (without removing rows)
            Keeps cancelled orders and users with no completed orders
            0 revenue is contributed, without missing rows

            SELECT
            user_id,
            SUM(
                CASE
                WHEN status = 'completed' THEN revenue
                ELSE 0
                END
            ) AS completed_revenue
            FROM orders
            GROUP BY user_id;

    CASE vs WHERE
        CASE
            Define what is counted
            Preserve population
            Compute rates safely
        WHERE
            Define who appears

Checkpoint
    Problem 1
        For each user, count
            Total orders
            Completed orders
            Output
                user_id
                total_orders
                completed_orders

        Answer
            SELECT
                user_id,
                COUNT(order_id),
                SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) as completed_orders
            FROM orders
            GROUP BY user_id;

    Problem 2
        For each user, calculate
            Total revenue
            Completed-order revenue
            Output
                user_id
                total_revenue
                completed_revenue

        Answer
            SELECT
                user_id,
                SUM(revenue) AS total_revenue,
                SUM(CASE WHEN status = 'completed' THEN revenue ELSE 0 END) as completed_orders
            FROM orders
            GROUP BY user_id;

    Problem 3
        For each country, calculate
            Total users
            Users with at least one completed order

        Answer
            

Visual Summary
    